## 1. Mount Google Drive and Load Data


In [ ]:
from google.colab import drive
from google.colab import output # Import output for explicit URL display

try:
    # Attempt to mount Google Drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Error mounting drive: {e}")
    # If mounting fails or doesn't show interactive prompt, try to display the auth URL directly
    print("If the interactive prompt did not appear, please copy and open this URL in your browser for authorization:")
    print(output.eval_js('google.colab.output.data.get("google.colab.drive-auth-url")'))


Mounted at /content/drive


Now, let's load the fomc_meeting_minutes1 file into a pandas DataFrame.

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/fomc_meeting_minutes1.csv' # Adjust path if necessary
df = pd.read_csv(file_path)

# Display the first few rows and info to confirm loading
print(df.head())
print(df.info())

   year   meeting_date_str meeting_date link_type  \
0  2022   January 05, 2022   2022-01-05      html   
1  2022  February 16, 2022   2022-02-16       pdf   
2  2022     April 06, 2022   2022-04-06       pdf   
3  2022       May 25, 2022   2022-05-25       pdf   
4  2022      July 06, 2022   2022-07-06       pdf   

                                                 url  \
0  https://www.federalreserve.gov/monetarypolicy/...   
1  https://www.federalreserve.gov/monetarypolicy/...   
2  https://www.federalreserve.gov/monetarypolicy/...   
3  https://www.federalreserve.gov/monetarypolicy/...   
4  https://www.federalreserve.gov/monetarypolicy/...   

                                        text_content          month  
0  Developments in Financial Markets and Open Mar...   January 2022  
1  Staff Review of the Economic Situation was clo...  February 2022  
2  Staff Review of the Economic Situation Availab...     April 2022  
3  Staff Review of the Economic Situation impleme...       May 2

## 2. Implement VADER Sentiment Analysis

Next, we'll perform VADER sentiment analysis on the DataFrame. This involves installing nltk and downloading necessary lexicons.

In [ ]:
# Install nltk if not already installed
try:
    import nltk
except ImportError:
    !pip install nltk
    import nltk

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import sent_tokenize
import nltk

# Download necessary NLTK data
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab') # Added to address LookupError for punkt_tab

sid = SentimentIntensityAnalyzer()

def score_document_vader(text):
    """Sentence-level aggregation — more stable for long documents"""
    if not isinstance(text, str): # Handle non-string values gracefully
        return {
            'vader_positive': 0.0,
            'vader_negative': 0.0,
            'vader_neutral': 1.0, # Neutral if text is not valid
            'vader_compound': 0.0,
        }
    sentences = sent_tokenize(text)
    if not sentences:
        return {
            'vader_positive': 0.0,
            'vader_negative': 0.0,
            'vader_neutral': 1.0, # Neutral if no sentences
            'vader_compound': 0.0,
        }
    scores = [sid.polarity_scores(s) for s in sentences]
    return {
        'vader_positive': sum(s['pos'] for s in scores) / len(scores),
        'vader_negative': sum(s['neg'] for s in scores) / len(scores),
        'vader_neutral': sum(s['neu'] for s in scores) / len(scores),
        'vader_compound': sum(s['compound'] for s in scores) / len(scores),
    }

# Apply to all meetings
vader_results = df['text_content'].apply(score_document_vader)
df = pd.concat([df, pd.DataFrame(vader_results.tolist())], axis=1)

print("VADER sentiment analysis applied. Displaying updated DataFrame head:")
display(df.head())

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


VADER sentiment analysis applied. Displaying updated DataFrame head:


,year,meeting_date_str,meeting_date,link_type,url,text_content,month,vader_positive,vader_negative,vader_neutral,vader_compound
0,2022,"January 05, 2022",2022-01-05,html,https://www.federalreserve.gov/monetarypolicy/...,Developments in Financial Markets and Open Mar...,January 2022,0.099875,0.034505,0.865630,0.218324
1,2022,"February 16, 2022",2022-02-16,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation was clo...,February 2022,0.093727,0.047259,0.859014,0.147391
2,2022,"April 06, 2022",2022-04-06,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation Availab...,April 2022,0.083168,0.053565,0.863267,0.118465
3,2022,"May 25, 2022",2022-05-25,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation impleme...,May 2022,0.091991,0.046082,0.861905,0.165242
4,2022,"July 06, 2022",2022-07-06,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation term in...,July 2022,0.082411,0.046636,0.870961,0.134674


## 3. Chart Visualization

Now, let's create the charts to visualize the sentiment trends.

### Chart 1: All 4 Polarity Trend Lines

This line chart will display Positive, Negative, Neutral, and Compound sentiment scores over time.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# Ensure 'date' column is datetime type from 'meeting_date'
df['date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('date')

fig = go.Figure()

fig.add_trace(go.Scatter(x=df['date'], y=df['vader_positive'], mode='lines', name='Positive', line=dict(color='green')))
fig.add_trace(go.Scatter(x=df['date'], y=df['vader_negative'], mode='lines', name='Negative', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=df['date'], y=df['vader_neutral'], mode='lines', name='Neutral', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=df['date'], y=df['vader_compound'], mode='lines', name='Compound', line=dict(color='purple')))

fig.update_layout(
    title_text='VADER Sentiment Polarity Trend Lines (Jan 2022 - Dec 2025)',
    xaxis_title='Date',
    yaxis_title='Sentiment Score',
    hovermode='x unified'
)

# Key events
annotations_data = {
    '2022-03-16': 'First Rate Hike',
    '2022-06-15': '75bp Hikes (starts)',
    '2022-11-02': '75bp Hikes (ends)',
    '2023-07-26': 'Peak Rate',
    '2024-09-18': 'First Cut (projected)'
}

for date_str, event_text in annotations_data.items():
    event_date = pd.to_datetime(date_str)

    # Add vertical line
    fig.add_shape(
        type="line",
        x0=event_date,
        y0=0,
        x1=event_date,
        y1=1,
        xref="x",
        yref="paper",
        line=dict(
            color="grey",
            width=1,
            dash="dot"
        )
    )

    # Add annotation text separately
    fig.add_annotation(
        x=event_date,
        y=1, # Position at the top of the plot
        yref="paper",
        text=event_text,
        showarrow=False,
        textangle=-90,
        font=dict(
            size=10,
            color="darkred"
        )
    )

fig.show()

### Discussion for Chart 1: All 4 Polarity Trend Lines

The chart above displays the trends for positive, negative, neutral, and compound sentiment scores over the 2022/2025 period. We can observe how different events influenced the overall sentiment in the FOMC meeting minutes.

Neutral Sentiment Explanation:

The Neutral sentiment score consistently hovers near 0.83–0.87 because VADER is designed to classify text into positive, negative, or neutral categories. In formal documents like FOMC meeting minutes, a large portion of the text is factual and does not carry strong emotional valence. These parts are typically categorized as neutral by VADER. Even sentences with some positive or negative words might still have a high neutral score if the overall sentiment isn't overwhelmingly positive or negative. The formal and careful language used in such documents inherently leads to a high proportion of neutral sentences.


### Chart 2: Compound Sentiment Time Series with Rolling Average and Economic Regimes

This chart focuses solely on the compound sentiment score, with a 3-month rolling average, and highlights different economic regimes.

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Ensure 'date' column is datetime type from 'meeting_date'
df['date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('date')

# Calculate 3-month rolling average for compound sentiment
df['compound_rolling_avg'] = df['vader_compound'].rolling(window=3, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(x=df['date'], y=df['vader_compound'], mode='lines', name='Compound Sentiment', line=dict(color='purple'), opacity=0.6))
fig.add_trace(go.Scatter(x=df['date'], y=df['compound_rolling_avg'], mode='lines', name='3-Month Rolling Average', line=dict(color='darkviolet', width=2)))

fig.update_layout(
    title_text='Compound Sentiment Time Series with 3-Month Rolling Average',
    xaxis_title='Date',
    yaxis_title='Compound Sentiment Score',
    hovermode='x unified'
)

# Define economic regimes for shading
regimes = {
    'Aggressive Tightening': ('2022-01-01', '2022-12-31', 'lightblue'),
    'Plateau': ('2023-01-01', '2023-12-31', 'lightgreen'),
    'Pivot & Cuts': ('2024-01-01', '2025-12-31', 'lightcoral')
}

# Add shaded economic regimes as vertical rectangles
for regime, (start, end, color) in regimes.items():
    fig.add_vrect(
        x0=start,
        x1=end,
        fillcolor=color,
        opacity=0.3,
        layer="below",
        line_width=0,
        annotation_text=regime,
        annotation_position="top left"
    )

fig.show()

### Discussion for Chart 2: Compound Sentiment Time Series

The compound sentiment score, smoothed by a 3-month rolling average, provides a clearer view of the overall sentiment shifts. The shaded regions represent different economic regimes:

*   **Aggressive Tightening (2022):** During this period, we might expect to see a relatively lower or declining compound sentiment as the Fed aggressively raised rates to combat inflation. This could reflect concerns about economic slowdown or recession.
*   **Plateau (2023):** As the Fed paused rate hikes and rates peaked, sentiment might stabilize. This period could show a less volatile compound score, potentially reflecting a wait-and-see approach or a mixed outlook.
*   **Pivot & Cuts (2024–2025):** With projected rate cuts, compound sentiment might begin to improve, indicating optimism about economic recovery or a 'soft landing'.

**Does compound sentiment track market expectations or lag them?**

Analyzing the chart in conjunction with actual market movements and Fed communications is crucial. Often, financial markets are forward-looking and react to expectations of future Fed policy. FOMC minutes, however, reflect the past discussions and decisions of the committee. Therefore, it's common for sentiment derived from meeting minutes to lag market expectations. Markets might anticipate a policy shift based on economic data, while the minutes only confirm or elaborate on those decisions after they have been made or discussed internally. However, if the minutes contain unexpected language, they can also influence or correct market expectations, causing a more immediate reaction.


### Chart 3: Multi-Panel: Month-by-Month Positive Sentiment

This multi-panel chart shows the positive sentiment score for each month.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Ensure 'date' column is datetime type from 'meeting_date' and sort
df['date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('date')

df['year_month'] = df['date'].dt.to_period('M')
monthly_positive = df.groupby('year_month')['vader_positive'].mean().reset_index()
monthly_positive['year_month_str'] = monthly_positive['year_month'].dt.strftime('%Y-%m')

# Determine the number of rows and columns for the subplot grid
num_months = len(monthly_positive)
num_cols = 4  # Example: 4 columns
num_rows = (num_months + num_cols - 1) // num_cols

# Create subplots
fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=monthly_positive['year_month_str'].tolist(),
    vertical_spacing=0.05,
    horizontal_spacing=0.03
)

# Determine max y-limit for consistent scaling
max_y = df['vader_positive'].max() * 1.1

# Highlight threshold for positive spikes (e.g., top 15% positive months)
highlight_threshold = df['vader_positive'].quantile(0.85)

for i, (idx, row) in enumerate(monthly_positive.iterrows()):
    r = i // num_cols + 1
    c = i % num_cols + 1

    bar_color = 'green'
    # Highlight panels where Positive spikes
    if row['vader_positive'] > highlight_threshold:
        bar_color = 'gold' # Use gold for highlighting

    fig.add_trace(
        go.Bar(
            x=[row['year_month_str']],
            y=[row['vader_positive']],
            marker_color=bar_color,
            showlegend=False,
            name=row['year_month_str'] # For hover text
        ),
        row=r, col=c
    )

    # Update y-axis to be consistent and hide x-axis labels
    fig.update_yaxes(range=[0, max_y], title_text='Positive Score', row=r, col=c)
    fig.update_xaxes(showticklabels=False, row=r, col=c)

fig.update_layout(
    title_text='Month-by-Month Positive Sentiment',
    height=num_rows * 250, # Adjust height based on number of rows
    width=num_cols * 250,  # Adjust width based on number of columns
    showlegend=False,
    hovermode='x unified'
)

fig.show()

### Discussion for Chart 3: Month-by-Month Positive Sentiment

This multi-panel view allows us to see specific months where positive sentiment was higher. Spikes in positive sentiment (highlighted in yellow) in late 2023 – 2024 could indeed correspond to growing optimism about a 'soft landing' for the economy. This is particularly interesting if unemployment remained low despite the aggressive monetary tightening.

If the VADER positive scores indeed show an upward trend or spikes during periods when unemployment remained remarkably low despite significant rate hikes, it suggests that the Fed's communications might have become more confident in achieving its dual mandate (price stability and maximum employment) without triggering a deep recession. The ability to maintain a strong labor market while battling inflation could be perceived as a successful policy trajectory, leading to more optimistic language in the meeting minutes.


### Chart 4: Multi-Panel: Month-by-Month Negative Sentiment

Similar to Chart 3, this multi-panel chart illustrates monthly negative sentiment.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Ensure 'date' column is datetime type from 'meeting_date'
df['date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('date')

# Ensure 'year_month' is created (it's created in a previous cell, but adding here for robustness)
df['year_month'] = df['date'].dt.to_period('M')

monthly_negative = df.groupby('year_month')['vader_negative'].mean().reset_index()
monthly_negative['year_month_str'] = monthly_negative['year_month'].dt.strftime('%Y-%m')

# Determine the number of rows and columns for the subplot grid
num_months = len(monthly_negative)
num_cols = 4
num_rows = (num_months + num_cols - 1) // num_cols

# Create subplots
fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=monthly_negative['year_month_str'].tolist(),
    vertical_spacing=0.05,
    horizontal_spacing=0.03
)

# Determine max y-limit for consistent scaling
max_y = df['vader_negative'].max() * 1.1

# Highlight threshold for negative spikes
highlight_threshold = df['vader_negative'].quantile(0.85)

for i, (idx, row) in enumerate(monthly_negative.iterrows()):
    r = i // num_cols + 1
    c = i % num_cols + 1

    bar_color = 'blue'
    # Highlight panels where Negative spikes
    if row['vader_negative'] > highlight_threshold:
        bar_color = 'red'

    fig.add_trace(
        go.Bar(
            x=[row['year_month_str']],
            y=[row['vader_negative']],
            marker_color=bar_color,
            showlegend=False,
            name=row['year_month_str']
        ),
        row=r, col=c
    )

    # Update y-axis to be consistent and hide x-axis labels
    fig.update_yaxes(range=[0, max_y], title_text='Negative Score', row=r, col=c)
    fig.update_xaxes(showticklabels=False, row=r, col=c)

fig.update_layout(
    title_text='Month-by-Month Negative Sentiment',
    height=num_rows * 250, # Adjust height based on number of rows
    width=num_cols * 250,  # Adjust width based on number of columns
    showlegend=False,
    hovermode='x unified'
)

fig.show()

### Discussion for Chart 4: Month-by-Month Negative Sentiment

This multi-panel chart for negative sentiment allows us to identify periods where concerns or pessimism were more pronounced in the FOMC discussions. Highlighting specific months like October–December 2022, March 2023 (SVB collapse, credit stress language), and mid-2023 (inflation uncertainty) provides concrete examples where external events or ongoing challenges translated into more negative language within the minutes.

An elevated VADER negative score can indeed correlate with Fed pause decisions, but the relationship is complex. When the Fed is considering a pause in rate hikes, it's often because economic conditions are showing signs of stress or vulnerability, or because previous tightening has started to have its desired effect. These conditions would naturally lead to more cautious and potentially negative language in the meeting minutes. Therefore, a spike in negative sentiment might precede or coincide with a pause as the committee weighs the risks of further tightening against the stability of the economy.

## 4. Latent Dirichlet Allocation (LDA)

This section outlines the steps for performing LDA on the FOMC meeting minutes to uncover underlying topics and analyze their trends over time.

### Step 1: Load Libraries

First, we'll import all the necessary libraries for text processing, LDA modeling, visualization, and data manipulation.

In [ ]:
# Ensure gensim and pyLDAvis are installed at the very beginning
!pip install gensim pyLDAvis

import gensim
import nltk
import pyLDAvis.gensim_models as gensim_models
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import re
import string
from collections import Counter
import sys

# Ensure NLTK data is downloaded
nltk.download('stopwords')
nltk.download('punkt')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 118.5 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



True

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Step 2 & 3: Build Custom Stop Words and Clean Text with Gensim

We will define a custom list of stop words by combining NLTK's English stopwords with additional domain-specific terms relevant to FOMC meetings. After defining these stop words, we will clean the text using gensim for tokenization, lowercasing, and punctuation removal, and then filter out the stop words.

In [ ]:
import re

nltk.download('stopwords', quiet=True)

# Base: NLTK English stopwords
stop_words = nltk.corpus.stopwords.words('english')

# FOMC Extra: Use the explicitly provided list and add other research-based terms
fomc_extra_stopwords = [
    'committee', 'federal', 'reserve', 'board', 'participants', 'members', 'staff',
    'meeting', 'noted', 'discussed', 'indicated', 'suggested', 'division'
]

# Add months as custom stopwords
months = ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december']
fomc_extra_stopwords.extend(months)

# Custom research-based additions
fomc_extra_stopwords.extend([
    'bank', 'banks', 'us', 'would', 'could', 'also', 'percent', 'fomc', 'monetary', 'policy', 'economy', 'economic',
    'financial', 'market', 'markets', 'report', 'staffs', 'well', 'time', 'data',
    'continue', 'outlook', 'developments', 'information', 'condition', 'conditions',
    'pdf', 'minutes', 'deputy',
    'ing', 'con', 'tion', 'com', 'ment', 'cre', 'text', 'tions',
    'o_p_e_n', 'm_a_rk_e_t', 'page', 'open', 'review', 'manager', 'managers', 'committees','situation', # Explicitly add observed noise fragments
    'repo', 'tariffs', 'roughly', 'projections', 'standards',
    'chair', 'chairman', 'vice',
    'powell', 'williams', 'barkin', 'bostic', 'bullard', 'daly', 'kashkari', 'mester', 'george', # Common FOMC names/titles
    'jerome', 'lael', 'christopher', 'philip', 'michael', 'michelle', 'lisa',
    'governor', 'president', 'mrs', 'mr', 'ms', 'dr', # Common titles
    'expected', 'view', 'current', 'recent', 'over', 'year', 'past', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten'
    'division', 'director', 'senior', 'affairs', 'associate',
    'adviser', 'advisor', 'special', 'economist', 'statistics',
    'research', 'manager', 'assistant', 'deputy', 'secretary',
    'officer', 'president', 'governor', 'chair', 'vice', #Staff titles
    'ticipants', 'tees', 'par', 'flation', 'par',
    'ment', 'ing', 'tion'  # pdf artefacts
    'private', 'standing', 'attended', 'half',
    'office', 'systems', 'preferred', 'percentage',
    'soma', 'srf', 'runoff', 'portfolio', 'ample',
    'purchases', 'asset', 'government', 'lowering',
    'ukraine', 'russian', 'china', 'invasion',
    'infla', 'nearterm', 'bankingsector'
])

# Ensure all stopwords are lowercase and unique
stop_words.extend(fomc_extra_stopwords)
stop_words = list(set([word.lower() for word in stop_words])) # Make unique and lowercase

def preprocess_text(text):
    if not isinstance(text, str):
        return []

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove URLs (common in web-scraped text like FOMC minutes)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Remove email addresses
    text = re.sub(r'\S*@\S*\s?', '', text)

    # 4. Remove newline characters and extra spaces
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Remove common titles/honorifics that might precede names
    text = re.sub(r'\b(mr|ms|dr|chairman|chairwoman|governor|president|vice)\b', '', text)

    # 6. Remove any character that is not a letter or space.
    # This helps with artifacts like o__p_e_n_ and m__a_rk_e_t_
    text = re.sub(r'[^a-z\s]', '', text)

    # 7. Gensim's simple_preprocess for tokenization.
    tokens = gensim.utils.simple_preprocess(text, deacc=True, min_len=3) # min_len=3 to ensure words are at least 3 chars

    # 8. Remove stopwords
    tokens = [token for token in tokens if token not in stop_words]

    return tokens

# Apply preprocessing to the 'text_content' column
df['tokenized_text'] = df['text_content'].apply(preprocess_text)

print("Text cleaning and stop word removal complete. Displaying first few tokenized texts:")
display(df[['meeting_date_str', 'tokenized_text']].head())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Text cleaning and stop word removal complete. Displaying first few tokenized texts:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,meeting_date_str,tokenized_text
0,"January 05, 2022","[operations, turned, first, discussion, period..."
1,"February 16, 2022","[close, average, available, real, pce, unchang..."
2,"April 06, 2022","[available, indicators, growth, business, avai..."
3,"May 25, 2022","[implement, plan, balance, sheet, available, d..."
4,"July 06, 2022","[term, investments, environment, participation..."


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 4: Build Gensim Dictionary

We will create a dictionary from our tokenized texts, then filter out words that appear in too few or too many documents. This helps to reduce noise and improve the quality of the topic models.

In [ ]:
# Create Dictionary
# The gensim.corpora.Dictionary method creates a mapping from words to their integer ids.
dictionary = gensim.corpora.Dictionary(df['tokenized_text'])

# Report vocabulary size before filtering
print(f"Vocabulary size before filtering: {len(dictionary)}")

# Filter out words that appear in less than 3 documents (no_below=3)
# or in more than 85% of the documents (no_above=0.85).
# keep_n=10000 ensures we keep a maximum of 10000 most frequent words after filtering,
# which is a common practice to manage dictionary size.
dictionary.filter_extremes(no_below=3, no_above=0.85, keep_n=10000)

# Report vocabulary size after filtering
print(f"Vocabulary size after filtering: {len(dictionary)}")

print("Gensim dictionary built and filtered.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Vocabulary size before filtering: 5549
Vocabulary size after filtering: 2390
Gensim dictionary built and filtered.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 5: Build BoW Corpus

We will convert the list of tokenized documents into a Bag-of-Words (BoW) corpus using the dictionary created in the previous step. This representation counts the occurrences of each word in every document. We'll also report the number of documents and the average number of unique tokens per document.

In [ ]:
# Convert document (list of words) into a Bag of Words (BoW) format
corpus = [dictionary.doc2bow(doc) for doc in df['tokenized_text']]

# Report number of documents
num_documents = len(corpus)
print(f"Number of documents in the corpus: {num_documents}")

# Calculate average unique tokens per document
# For each document in the corpus, get the number of unique tokens (length of the list of tuples)
unique_tokens_per_doc = [len(doc) for doc in corpus]
average_unique_tokens = sum(unique_tokens_per_doc) / num_documents if num_documents > 0 else 0
print(f"Average unique tokens per document: {average_unique_tokens:.2f}")

print("BoW Corpus built successfully.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Number of documents in the corpus: 33
Average unique tokens per document: 672.15
BoW Corpus built successfully.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 6: Coherence Search Loop (K= 2 to 15)

To find the optimal number of topics for our LDA model, we will train several LDA models, varying the number of topics (K) from 2 to 15. For each model, we will compute its coherence score using the 'c_v' metric. A higher coherence score generally indicates more interpretable topics.

We choose an upper bound of K=15 as it's a common heuristic for initial topic modeling exploration.

In [ ]:
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
import plotly.graph_objects as go

# Function to compute coherence values
def compute_coherence_values(dictionary, corpus, texts, limit, start=2, step=1):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics,
                         random_state=42, passes=10, alpha='auto', eta='auto')
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

# Set the upper limit for K (number of topics)
limit = 16 # K from 2 to 15
start = 2
step = 1

# Compute coherence values
model_list, coherence_values = compute_coherence_values(dictionary=dictionary, corpus=corpus, texts=df['tokenized_text'], limit=limit, start=start, step=step)

# Plotting the coherence scores
x = range(start, limit, step)
fig = go.Figure(
    data=[go.Scatter(x=list(x), y=coherence_values, mode='lines+markers', name='Coherence Score')],
    layout=go.Layout(
        title=go.layout.Title(text='Coherence Score vs. Number of Topics (K)'),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text='Number of Topics (K)')),
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text='Coherence Score (c_v)'))
    )
)

fig.show()

print("Coherence search loop completed and plot displayed. Analyze the plot to identify the optimal number of topics.")

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is

Coherence search loop completed and plot displayed. Analyze the plot to identify the optimal number of topics.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 7: Train Final LDA at best K

Based on the coherence plot, we will select the optimal number of topics and train the final LDA model. We will then print the top-10 words for each topic to understand their themes and assign meaningful human-readable labels.

In [ ]:
import numpy as np

# Find the optimal number of topics based on the highest coherence score
optimal_k_index = np.argmax(coherence_values)
optimal_num_topics = range(start, limit, step)[optimal_k_index]

# Re-train LDA model at optimal K with specified parameters
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=optimal_num_topics,
                     random_state=42, passes=15, alpha='auto', eta='auto')

print(f"Final LDA Model trained with {optimal_num_topics} topics.\n")

# Print top-10 words for each topic
topic_words = []
for i, topic in lda_model.show_topics(formatted=False, num_words=10):
    words = [word[0] for word in topic]
    topic_words.append(words)
    print(f"Topic {i+1}: {', '.join(words)}")

topic_labels = {
    i: f"Topic {i+1}" for i in range(optimal_num_topics)
}

print("\nAssigned human-readable topic labels.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Final LDA Model trained with 8 topics.

Topic 1: pandemic, accommodative, fourth, normalization, omicron, virus, accommodation, covid, bottlenecks, variant
Topic 2: moving, cent, tive, robust, weigh, cmbs, nomic, eco, ployment, expect
Topic 3: moving, private, sustainably, tightening, highly, achieving, fourth, slowing, confidence, slow
Topic 4: tightening, banking, tighter, robust, cumulative, third, weigh, slowing, raise, lags
Topic 5: private, tariff, finance, vulnerabilities, almost, relatively, supported, adjustments, considering, policies
Topic 6: ongoing, size, related, firming, raise, imbalances, disruptions, highly, war, chain
Topic 7: tightening, war, far, ongoing, related, raise, imbalances, robust, pandemic, reduction
Topic 8: tightening, third, slow, related, pandemic, robust, private, ongoing, highly, moving

Assigned human-readable topic labels.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 8: Add LDA_topics+LDA_Words columns


In [ ]:
def format_topics_sentences(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get main topic in each document
    for i, row_list in enumerate(ldamodel[corpus]):
        row_list = sorted(row_list, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Percent contribution and Keywords for each document
        topic_info = []
        topic_words_list = []
        for j, (topic_num, prop_topic) in enumerate(row_list):
            if j < 3:  # Get top 3 topics
                wp = ldamodel.show_topic(topic_num, topn=5)
                topic_keywords = ", ".join([word for word, prop in wp])
                topic_label = topic_labels.get(topic_num, f"Topic {topic_num+1}")
                topic_info.append(f"{topic_label} ({prop_topic*100:.1f}%)")
                topic_words_list.append(f"{topic_label.split(' ')[0][0]}{topic_num+1}: {topic_keywords}")

        sent_topics_df = pd.concat([
            sent_topics_df,
            pd.DataFrame([{
                'Dominant_Topic_Num': row_list[0][0],
                'Dominant_Topic_Perc': row_list[0][1],
                'LDA_Topics': "; ".join(topic_info),
                'LDA_Words': " | ".join(topic_words_list)
            }])
        ], ignore_index=True)

    # Add original text to the output
    contents = pd.Series(texts)
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)

    return sent_topics_df

# Apply the function to get the topics and keywords
df_topic_sents_keywords = format_topics_sentences(ldamodel=lda_model, corpus=corpus, texts=df['text_content'])

# Name the columns
df_topic_sents_keywords.columns = ['Dominant_Topic_Num', 'Dominant_Topic_Perc', 'LDA_Topics', 'LDA_Words', 'Text_Content']

# Merge these new columns back into the original DataFrame
df = pd.concat([df, df_topic_sents_keywords[['Dominant_Topic_Num', 'Dominant_Topic_Perc', 'LDA_Topics', 'LDA_Words']]], axis=1)

print("LDA topics and keywords added to DataFrame. Displaying updated DataFrame head:")
display(df.head())


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

LDA topics and keywords added to DataFrame. Displaying updated DataFrame head:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,year,meeting_date_str,meeting_date,link_type,url,text_content,month,vader_positive,vader_negative,vader_neutral,vader_compound,date,compound_rolling_avg,year_month,tokenized_text,Dominant_Topic_Num,Dominant_Topic_Perc,LDA_Topics,LDA_Words
0,2022,"January 05, 2022",2022-01-05,html,https://www.federalreserve.gov/monetarypolicy/...,Developments in Financial Markets and Open Mar...,January 2022,0.099875,0.034505,0.865630,0.218324,2022-01-05,0.218324,2022-01,"[operations, turned, first, discussion, period...",0,0.999830,Topic 1 (100.0%),"T1: pandemic, accommodative, fourth, normaliza..."
1,2022,"February 16, 2022",2022-02-16,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation was clo...,February 2022,0.093727,0.047259,0.859014,0.147391,2022-02-16,0.182857,2022-02,"[close, average, available, real, pce, unchang...",0,0.999786,Topic 1 (100.0%),"T1: pandemic, accommodative, fourth, normaliza..."
2,2022,"April 06, 2022",2022-04-06,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation Availab...,April 2022,0.083168,0.053565,0.863267,0.118465,2022-04-06,0.161393,2022-04,"[available, indicators, growth, business, avai...",5,0.999753,Topic 6 (100.0%),"T6: ongoing, size, related, firming, raise"
3,2022,"May 25, 2022",2022-05-25,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation impleme...,May 2022,0.091991,0.046082,0.861905,0.165242,2022-05-25,0.143700,2022-05,"[implement, plan, balance, sheet, available, d...",5,0.999784,Topic 6 (100.0%),"T6: ongoing, size, related, firming, raise"
4,2022,"July 06, 2022",2022-07-06,pdf,https://www.federalreserve.gov/monetarypolicy/...,Staff Review of the Economic Situation term in...,July 2022,0.082411,0.046636,0.870961,0.134674,2022-07-06,0.139461,2022-07,"[term, investments, environment, participation...",5,0.999788,Topic 6 (100.0%),"T6: ongoing, size, related, firming, raise"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 9: Graph 1: Topic-Word Distribution (Plotly)

This chart will display horizontal bar charts for each topic, showing the top-N words and their probability weights.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Number of top words to display per topic
num_top_words = 10

# Create subplots for each topic
fig_words = make_subplots(rows=optimal_num_topics, cols=1,
                        subplot_titles=[topic_labels[i] for i in range(optimal_num_topics)],
                        vertical_spacing=0.07) # Adjusted vertical_spacing

for i, topic in enumerate(lda_model.show_topics(formatted=True, num_words=num_top_words)):
    topic_id = topic[0]
    topic_terms = topic[1]

    # Parse topic_terms string into words and probabilities
    words_probs = []
    for item in topic_terms.split(' + '):
        prob, word = item.split('*')
        words_probs.append((word.strip().replace('"', ''), float(prob)))

    # Sort words by probability in descending order
    words_probs.sort(key=lambda x: x[1], reverse=True)

    words = [wp[0] for wp in words_probs]
    probs = [wp[1] for wp in words_probs]

    # Create horizontal bar chart
    fig_words.add_trace(go.Bar(
        y=words,
        x=probs,
        orientation='h',
        marker_color='lightblue',
        name=topic_labels[topic_id],
        hovertemplate='<b>Word</b>: %{y}<br><b>Probability</b>: %{x:.3f}<extra></extra>'
    ), row=i+1, col=1)

    # Reverse axes for better readability (highest prob at top)
    fig_words.update_yaxes(autorange='reversed', row=i+1, col=1)

fig_words.update_layout(
    title_text='Topic-Word Distribution',
    height=400 * optimal_num_topics,
    showlegend=False
)

fig_words.show()

# Export as HTML
fig_words.write_html("topic_word_distribution.html")
print("Topic-Word Distribution chart generated and exported as 'topic_word_distribution.html'.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Topic-Word Distribution chart generated and exported as 'topic_word_distribution.html'.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 10: Graph 2: Document-Topic Heatmap (Plotly)

This heatmap will show meeting months on one axis and topics on the other, with color indicating the topic weight.

In [ ]:
import plotly.express as px

# Prepare data for heatmap
# Get topic distribution for each document
def get_document_topic_distribution(ldamodel, corpus):
    doc_topic_probs = []
    for i, row_list in enumerate(ldamodel[corpus]):
        # Initialize topic probabilities for the current document to zero
        probs = [0.0] * ldamodel.num_topics
        for topic_num, prop_topic in row_list:
            probs[topic_num] = prop_topic
        doc_topic_probs.append(probs)
    return pd.DataFrame(doc_topic_probs)

df_topic_distribution = get_document_topic_distribution(lda_model, corpus)

# Assign column names using topic labels
df_topic_distribution.columns = [topic_labels.get(i, f'Topic {i+1}') for i in range(lda_model.num_topics)]

# Ensure 'year_month_str' exists in df for consistent merging and plotting
# df['year_month'] is of type Period, convert it to string for plotting
df['year_month_str'] = df['year_month'].dt.strftime('%Y-%m')

# Add meeting dates (or months) to the dataframe
df_topic_distribution['meeting_date'] = df['date']
df_topic_distribution['year_month'] = df['year_month_str']

# Sort by date for proper time series display
df_topic_distribution = df_topic_distribution.sort_values(by='meeting_date').reset_index(drop=True)

# Create the heatmap
fig_heatmap = px.imshow(
    df_topic_distribution.drop(columns=['meeting_date', 'year_month']).T, # Transpose to have topics as rows, documents as columns
    x=df_topic_distribution['year_month'],
    y=df_topic_distribution.columns.drop(['meeting_date', 'year_month']),
    color_continuous_scale='Viridis',
    labels=dict(x="Meeting Month", y="Topic", color="Topic Proportion"),
    title="Document-Topic Heatmap Over Time"
)

fig_heatmap.update_xaxes(side="top")
fig_heatmap.update_layout(
    height=500,
    width=1000
)

fig_heatmap.show()

print("Document-Topic Heatmap generated.")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Document-Topic Heatmap generated.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 11: Graph 3: Topic Trends Over Time (Stacked Area)

This stacked area chart will visualize the proportion of each topic over time, showing how the focus of discussions shifts. Each colored area will represent a topic, with its height indicating its relative prevalence in the meeting minutes for a given month.

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Ensure 'year_month' is in datetime format for proper sorting and plotting
df_topic_distribution['year_month_dt'] = pd.to_datetime(df_topic_distribution['year_month'])
df_topic_distribution = df_topic_distribution.sort_values(by='year_month_dt').reset_index(drop=True)

# Melt the DataFrame to long format for Plotly Express stacked area chart
# Exclude meeting_date, year_month, and year_month_dt as they are not topics
topic_columns = [col for col in df_topic_distribution.columns if 'Topic' in col or 'Inflation' in col or 'Labor' in col or 'Financial' in col or 'International' in col or 'Monetary' in col]

df_melted = df_topic_distribution.melt(id_vars=['year_month_dt'], value_vars=topic_columns,
                                      var_name='Topic', value_name='Proportion')

fig_stacked_area = px.area(df_melted,
                           x='year_month_dt',
                           y='Proportion',
                           color='Topic',
                           title='Topic Trends Over Time (Stacked Area Chart)',
                           labels={'year_month_dt': 'Date', 'Proportion': 'Topic Proportion'},
                           hover_name='Topic')

fig_stacked_area.update_layout(xaxis_title='Date', yaxis_title='Topic Proportion')
fig_stacked_area.show()

print("Topic Trends Over Time (Stacked Area Chart) generated.")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Topic Trends Over Time (Stacked Area Chart) generated.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Step 12: Graph 4: Top Topic per Month + Federal Funds Rate

This chart combines the dominant topic for each meeting with the historical Federal Funds Rate, allowing us to visualize potential correlations between policy decisions and the focus of FOMC discussions.

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Calculer le topic dominant par réunion
topic_distributions = []
for bow in corpus:
    dist = dict(lda_model.get_document_topics(bow, minimum_probability=0.0))
    topic_distributions.append([dist.get(k, 0.0) for k in range(lda_model.num_topics)])

tpm = np.array(topic_distributions)
dominant_topics = np.argmax(tpm, axis=1)          # index 0-based
dominant_probs  = np.max(tpm, axis=1)

# Add to dataframe
df['dominant_topic'] = dominant_topics + 1        # 1-based pour affichage
df['dominant_prob']  = dominant_probs

# 2. Federal Funds Rate — hardcoded
fed_funds_data = {
    '2022-01-26': 0.25, '2022-03-16': 0.50, '2022-05-04': 1.00,
    '2022-06-15': 1.75, '2022-07-27': 2.50, '2022-09-21': 3.25,
    '2022-11-02': 4.00, '2022-12-14': 4.50,
    '2023-02-01': 4.75, '2023-03-22': 5.00, '2023-05-03': 5.25,
    '2023-06-14': 5.25, '2023-07-26': 5.50, '2023-09-20': 5.50,
    '2023-11-01': 5.50, '2023-12-13': 5.50,
    '2024-01-31': 5.50, '2024-03-20': 5.50, '2024-05-01': 5.50,
    '2024-06-12': 5.50, '2024-07-31': 5.50, '2024-09-18': 5.00,
    '2024-11-07': 4.75, '2024-12-18': 4.50,
    '2025-01-29': 4.50, '2025-03-19': 4.50, '2025-05-07': 4.50,
    '2025-06-18': 4.50, '2025-07-30': 4.50, '2025-09-17': 4.50,
    '2025-10-29': 4.50, '2025-12-10': 4.50,
}

# Convert 'meeting_date' in df to datetime objects if not already and sort
df['meeting_date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values(by='meeting_date').reset_index(drop=True)

# Create a DataFrame for Federal Funds Rate announcement dates and values
fed_funds_announcements = pd.DataFrame(list(fed_funds_data.items()), columns=['date_announcement_str', 'fed_rate'])
fed_funds_announcements['date_announcement'] = pd.to_datetime(fed_funds_announcements['date_announcement_str'])
fed_funds_announcements = fed_funds_announcements.sort_values(by='date_announcement').reset_index(drop=True)

# Use merge_asof to get the latest Fed Funds Rate at or before each meeting date
df = pd.merge_asof(df,
                   fed_funds_announcements[['date_announcement', 'fed_rate']],
                   left_on='meeting_date',
                   right_on='date_announcement',
                   direction='backward',
                   suffixes=('', '_announcement'))

# Drop the temporary announcement date column if it was added by merge_asof
df = df.drop(columns=['date_announcement'], errors='ignore')

# 3. Colors
topic_colors = {
    1: '#636EFA', 2: '#EF553B', 3: '#00CC96', 4: '#AB63FA',
    5: '#FFA15A', 6: '#19D3F3', 7: '#FF6692', 8: '#B6E880',
}

# 4. Bar chart
fig = go.Figure()

for t in sorted(df['dominant_topic'].unique()):
    mask = df['dominant_topic'] == t
    label = topic_labels.get(t - 1, f'Topic {t}')

    fig.add_trace(go.Bar(
        x=df.loc[mask, 'meeting_date'],
        y=df.loc[mask, 'dominant_prob'],
        name=label,
        marker_color=topic_colors.get(t, '#888888'),
        hovertemplate=(
            '<b>%{x|%b %Y}</b><br>'
            f'Topic {t}: {label}<br>'
            'Poids : %{y:.3f}<extra></extra>'
        )
    ))

# 5. Overlay taux Fed (axe secondaire)
fig.add_trace(go.Scatter(
    x=fed_funds_announcements['date_announcement'], # Use announcement dates for rate trend
    y=fed_funds_announcements['fed_rate'],
    name='Fed Funds Rate (%)',
    mode='lines+markers',
    line=dict(color='black', width=2.5, dash='dot'),
    marker=dict(size=7, symbol='diamond'),
    yaxis='y2',
    hovertemplate='<b>%{x|%b %Y}</b><br>Taux : %{y:.2f}%<extra></extra>'
))

# 6. Layout
fig.update_layout(
    title=dict(
        text='Dominant LDA Topic per FOMC Meeting + Federal Funds Rate (2022–2025)',
        font=dict(size=16)
    ),
    barmode='stack',
    xaxis=dict(title='Meeting Date', tickformat='%b %Y'),
    yaxis=dict(title='Topic Weight', range=[0, 1]),
    yaxis2=dict(
        title='Federal Funds Rate (%)',
        overlaying='y',
        side='right',
        range=[0, 7],
        showgrid=False,
        tickfont=dict(color='black'),
        titlefont=dict(color='black')
    ),
    legend=dict(
        orientation='v',
        x=1.08, y=1,
        font=dict(size=9)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=550,
    width=1200,
)

fig.show()
fig.write_html('graph4_top_topic_per_month.html')
print('Graph 4 saved: graph4_top_topic_per_month.html')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Graph 4 saved: graph4_top_topic_per_month.html


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 13: Interactive Topic Visualization (pyLDAvis)

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Prepare visualization
vis = pyLDAvis.gensim_models.prepare(
    lda_model,
    corpus,
    dictionary,
    mds='mmds',
    sort_topics=False
)

html_str = pyLDAvis.prepared_data_to_html(vis)
display(HTML(html_str))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

### Step 14: Save Final DataFrame

In [ ]:
# Save the enriched DataFrame to a CSV file
output_path = '/content/drive/MyDrive/fomc_minutes_with_sentiment_and_topics.csv'
df.to_csv(output_path, index=False)

print(f"Enriched DataFrame saved to {output_path}")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Enriched DataFrame saved to /content/drive/MyDrive/fomc_minutes_with_sentiment_and_topics.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v